# CVIU v3 Experiments Runner (`Colab Pro+ A100`)

请按以下步骤一键启动全套新实验：
1. 菜单栏点击 **Runtime → Change runtime type → 选择 `A100 GPU`**；
2. 点击 **Runtime → Run all (`Ctrl+F9`)**（首次运行会弹一次 Google Drive 授权窗，点允许即可）；
3. 每个 Stage 跑完后都会立即将 JSON / JSONL 结果镜像到 `UltraSync/ConvertAsset/remote_runs/cviu_v3_20261116/`；
4. **Cell 5** 跑完后会自动 flush Google Drive 并**自动断开 A100 运行时（`AUTO_DISCONNECT = True`）**，绝不挂机空烧算力。

In [ ]:
import subprocess, sys, torch, transformers

if not torch.cuda.is_available():
    raise RuntimeError('当前为 CPU 运行时！请在 Runtime -> Change runtime type 中切换为 A100 GPU 后再运行。')

for idx in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(idx)
    print(f'GPU[{idx}]: {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GiB')
print('torch:', torch.__version__, '| transformers:', transformers.__version__)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'qwen-vl-utils', 'einops', 'accelerate'], check=True)
print('Environment & lightweight dependencies ready.')

In [ ]:
import os, pathlib, subprocess
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
ultrasync = pathlib.Path('/content/drive/MyDrive/UltraSync')
sync_py = ultrasync / '_portable_skill/ultra-cloud-suite/scripts/sync_embodied_repos.py'

repo_dir = pathlib.Path('/content/dev/ConvertAsset')
if not (repo_dir / '.git').exists():
    subprocess.run([
        'python3', str(sync_py), 'pull',
        '--drive-dir', str(ultrasync),
        '--target-dev-root', '/content/dev',
        '--repos', 'ConvertAsset',
        '--apply-bundle', '--apply-overlay',
    ], check=True)

overlay_tgz = ultrasync / 'ConvertAsset/remote_runs/cviu_v3_20261116/cviu_v3_code_overlay.tar.gz'
subprocess.run(['tar', '-xzf', str(overlay_tgz), '-C', str(repo_dir)], check=True)
print('Overlayed CVIU v3 code package onto', repo_dir)

dev_link = pathlib.Path('/cpfs/user/zhuzihou/dev')
dev_link.parent.mkdir(parents=True, exist_ok=True)
if dev_link.is_symlink():
    dev_link.unlink()
if not dev_link.exists():
    dev_link.symlink_to('/content/dev')

assets_cviu = pathlib.Path('/cpfs/user/zhuzihou/assets/convertasset_research/experiments/cviu')
assets_cviu.mkdir(parents=True, exist_ok=True)
if not (assets_cviu / 'formal_selected_captures_v1').is_dir():
    subprocess.run(['tar', '-xzf', str(ultrasync / 'ConvertAsset/archives/cviu_formal_and_pilot_inputs_v1.tar.gz'), '-C', str(assets_cviu)], check=True)
if not (assets_cviu / 'formal_model_runs_v1').is_dir():
    subprocess.run(['tar', '-xzf', str(ultrasync / 'ConvertAsset/archives/cviu_formal_model_runs_v1.tar.gz'), '-C', str(assets_cviu)], check=True)

os.environ['EMBODIED_DEV_ROOT'] = '/content/dev'
os.environ['CVIU_ASSETS_ROOT'] = str(assets_cviu)
print('Workspace & frozen 719.6 MB inputs unpacked and verified at', assets_cviu)

In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable, '-m', 'unittest', 'discover',
    '-s', '/content/dev/ConvertAsset/tests',
    '-p', 'test_cviu_v3_pipeline.py', '-v',
], check=True)

In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable, '-u',
    '/content/dev/ConvertAsset/paper/shared/evidence/experiments/14_cviu_v3/run_colab_cviu_v3_all.py',
    '--stages', '1,2,3,4',
], check=True)

In [ ]:
import json, pathlib
from google.colab import drive

AUTO_DISCONNECT = True
analysis_dir = pathlib.Path('/content/drive/MyDrive/UltraSync/ConvertAsset/remote_runs/cviu_v3_20261116/analysis')

for name in ('scaling_and_families.json', 'variance_decomposition.json', 'optical_mechanisms.json', 'risk_auditor.json'):
    p = analysis_dir / name
    print(f'\n=== {name} ===')
    if not p.is_file():
        print('  MISSING')
        continue
    d = json.loads(p.read_text(encoding='utf-8'))
    if name == 'scaling_and_families.json':
        for mk, mv in d.get('models', {}).items():
            ms = mv['metrics']
            print(f"  {mk:20s} | O={ms['original_hit_rate']['estimate']:.4f} C={ms['converted_hit_rate']['estimate']:.4f} Flip={ms['flip_rate']['estimate']:.4f} Loss={ms['loss_rate']['estimate']:.4f} Gain={ms['gain_rate']['estimate']:.4f}")
    elif name == 'variance_decomposition.json':
        print(' ', json.dumps(d, indent=2)[:1200])
    elif name == 'optical_mechanisms.json' and 'all_480_pairs' in d:
        sub = d['all_480_pairs']
        print('  flips_vs_O:', {k: round(v['estimate'], 4) for k, v in sub['flips_vs_original'].items() if v.get('estimate') is not None})
        print('  factorial:', {k: round(v['estimate'], 4) for k, v in sub['factorial_effects'].items() if v.get('estimate') is not None})
    elif name == 'risk_auditor.json' and 'targets' in d:
        for tk in ('any_8b_32b_loss', '8b_loss', '8b_flip'):
            if tk in d['targets'] and 'deploy_auto' in d['targets'][tk]:
                t10 = d['targets'][tk]['deploy_auto']['test_10_scenes']
                loso = d['targets'][tk]['deploy_auto']['loso_30_scenes']
                print(f"  {tk:16s} | test AUROC={t10['auroc']:.4f} R@20%={t10['top_20pct_recall']:.4f} | LOSO AUROC={loso['auroc']:.4f} R@20%={loso['top_20pct_recall']:.4f}")

drive.flush_and_unmount()
print('\nDrive flushed cleanly.')
if AUTO_DISCONNECT:
    from google.colab import runtime
    print('Unassigning Colab GPU runtime...')
    runtime.unassign()